# Téléchargement et traitement des données IMERG Final Run

Ce notebook télécharge les données IMERGv07 Final Run depuis le serveur de la NASA, les traite pour les ajuster à un domaine géographique donné, puis écrit les sorties au format GeoTIFF.

### Étapes :
1. Télécharger les fichiers depuis le serveur de la NASA pour une période donnée.
2. Traiter les fichiers téléchargés pour le domaine et leur appliquer le facteur d'échelle.
3. Enregistrer les données traitées au format GeoTIFF.

In [3]:
# Importer les bibliothèques nécessaires
import sys
import subprocess
import os
import datetime as DT
import osgeo.gdal as gdal
from osgeo.gdal import gdalconst
from osgeo.gdalconst import GA_ReadOnly
#from gdalconst import GA_ReadOnly
#from gdalconst import *
import numpy as np

Indiquez l'adresse électronique associée à votre compte GPM

In [ ]:

server = 'https://arthurhouhttps.pps.eosdis.nasa.gov/gpmdata'
file_prefix = '3B-HHR-GIS.MS.MRG.3IMERG.'
file_suffix = '.V06B.tif'
# Adresse électronique associée au compte PMM. Elle sert à accéder aux données. Créez un compte sur https://pmm.nasa.gov/data-access/downloads/gpm si vous n'en avez pas.
email = 'email-address@uiowa.edu'

Exécutez la cellule suivante pour télécharger les données, puis placez les fichiers dans le répertoire /data/precip/ de votre répertoire EF5. 

In [ ]:
def main(argv):
  # Coordonnées du domaine pour l'Afrique de l'Ouest
  xmin = -21.4
  xmax = 30.4
  ymin = -2.9
  ymax = 33.1

# pour les dates de début et de fin, on part de la date du pic de crue puis on ajoute une marge de 5 jours (la période commence donc 5 jours avant le pic et se termine 5 jours après)
# La date de début remonte ensuite 3 mois plus tôt, afin de fournir les données de la période de mise en régime (warm up) d'EF5
  # Date de début
  year_i = 2020
  month_i = 10
  day_i = 8

  # Date de fin
  year_f = 2020
  month_f = 10
  day_f = 11

  # parcourir la liste des fichiers et récupérer chacun d'eux
  initial_date = DT.datetime(year_i,month_i,day_i,0,0,0)
  final_date = DT.datetime(year_f,month_f,day_f,0,0,0)
  delta_time = DT.timedelta(minutes=30)

  # Boucle sur les dates
  current_date = initial_date
  acumulador_30M = 0

  while (current_date < final_date):
    initial_time_stmp = current_date.strftime('%Y%m%d-S%H%M%S')

    final_time = current_date + DT.timedelta(minutes=29)
    final_time_stmp = final_time.strftime('E%H%M59')
    final_time_gridout = current_date + DT.timedelta(minutes=30)

    folder = current_date.strftime('%Y/%m/%d')

    date_stamp = initial_time_stmp + '-' + final_time_stmp + '.' + '{:04d}'.format(acumulador_30M)

    filename = folder +"/gis/"+file_prefix + date_stamp + file_suffix
    print('Téléchargement de ' + server + '/' + filename)
    try:
        # Télécharger depuis le serveur de la NASA
        get_file(filename)

        # Traiter le fichier pour le domaine et le format attendu par EF5
        # Le nom de fichier porte l'horodatage final car il correspond au cumul jusqu'à cet instant
        gridOutName = 'imerg.' + final_time_gridout.strftime('%Y%m%d%H%M') + '.tif'
        local_filename = file_prefix + date_stamp + file_suffix
        NewGrid, nx, ny, gt, proj = processIMERG(local_filename,xmin,ymin,xmax,ymax)

        # Écrire le fichier traité
        WriteGrid(gridOutName, NewGrid, nx, ny, gt, proj)
        os.remove(local_filename)
        os.remove('./OutTemp.tif')
    except Exception as e:
        print(e)
        print(filename)

    # Avancer dans le temps
    current_date = current_date + delta_time

    # Si le jour change, réinitialiser acumulador_30M
    if (acumulador_30M < 1410):
      acumulador_30M = acumulador_30M + 30
    else:
      print("Nouveau jour")
      acumulador_30M = 0

def get_file(filename):
    ''' Récupérer le fichier indiqué depuis jsimpsonhttps à l'aide de curl. '''
    url = server + '/' + filename

    cmd = 'curl -sO -u ' + email + ':' + email + ' ' + url
    args = cmd.split()

    process = subprocess.Popen(args, stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    process.wait() # attendre afin que le programme ne se termine pas
                        # avant d'avoir récupéré tous les fichiers

def ReadandWarp(gridFile,xmin,ymin,xmax,ymax):
    #Lire la grille et l'ajuster à la grille du domaine
    #On suppose qu'aucune reprojection n'est nécessaire et que le SCR est EPSG:4326
    rawGridIn = gdal.Open(gridFile, GA_ReadOnly)

    # Ajuster la grille
    pre_ds = gdal.Translate('OutTemp.tif', rawGridIn, options="-co COMPRESS=Deflate -a_nodata 29999 -a_ullr -180.0 90.0 180.0 -90.0")

    gt = pre_ds.GetGeoTransform()
    proj = pre_ds.GetProjection()
    nx = pre_ds.GetRasterBand(1).XSize
    ny = pre_ds.GetRasterBand(1).YSize
    NoData = 29999
    pixel_size = gt[1]

    #Ajuster à la résolution et à l'emprise du modèle
    ds = gdal.Warp('', pre_ds, srcNodata=NoData, srcSRS='EPSG:4326', dstSRS='EPSG:4326', dstNodata='29999', format='VRT', xRes=pixel_size, yRes=-pixel_size, outputBounds=(xmin,ymin,xmax,ymax))

    WarpedGrid = ds.ReadAsArray()
    new_gt = ds.GetGeoTransform()
    new_proj = ds.GetProjection()
    new_nx = ds.GetRasterBand(1).XSize
    new_ny = ds.GetRasterBand(1).YSize

    return WarpedGrid, new_nx, new_ny, new_gt, new_proj

def WriteGrid(gridOutName, dataOut, nx, ny, gt, proj):
    #Écrit un GeoTIFF à partir des informations de géoréférencement contenues dans RefInfo
    driver = gdal.GetDriverByName('GTiff')
    dst_ds = driver.Create(gridOutName, nx, ny, 1, gdal.GDT_Float32, ['COMPRESS=DEFLATE'])
    dst_ds.SetGeoTransform(gt)
    dst_ds.SetProjection(proj)
    dataOut.shape = (-1, nx)
    dst_ds.GetRasterBand(1).WriteArray(dataOut, 0, 0)
    dst_ds.GetRasterBand(1).SetNoDataValue(-9999.0)
    dst_ds = None

def processIMERG(local_filename,llx,lly,urx,ury):
  # Traiter la grille
  # Lire et extraire la sous-grille
  NewGrid, nx, ny, gt, proj = ReadandWarp(local_filename,llx,lly,urx,ury)

  # Appliquer le facteur d'échelle
  NewGrid = NewGrid*0.1

  return NewGrid, nx, ny, gt, proj

if __name__ == '__main__':
    main(sys.argv)

New day
New day
New day


Si vous utilisez Colab, servez-vous de la cellule suivante pour compresser tous les fichiers GPM dans un .zip. Vous pourrez ensuite télécharger le fichier .zip sur votre machine locale.

In [4]:
import os
import shutil
from zipfile import ZipFile

# Créer le dossier GPM_precip s'il n'existe pas déjà
output_folder = "GPM_precip"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Dossier '{output_folder}' créé.")

# Rechercher et déplacer tous les fichiers imerg*.tif dans le dossier GPM_precip
for file in os.listdir("."):
    if file.startswith("imerg") and file.endswith(".tif"):
        shutil.move(file, os.path.join(output_folder, file))
        print(f"Fichier '{file}' déplacé vers le dossier '{output_folder}'.")

# Compresser le dossier GPM_precip dans un fichier ZIP
zip_filename = f"{output_folder}.zip"
with ZipFile(zip_filename, 'w') as zipf:
    for root, _, files in os.walk(output_folder):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_folder)
            zipf.write(file_path, arcname)
            print(f"Fichier '{file}' ajouté au ZIP.")

print(f"Dossier '{output_folder}' compressé dans '{zip_filename}'.")


Folder 'GPM_precip' created.
File 'imerg.202010100530.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010101930.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010101530.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010101500.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010100300.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010102200.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010102300.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010100700.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010110000.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010100030.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010101730.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010100430.30minAccum.tif' moved to the folder 'GPM_precip'.
File 'imerg.202010100830.30minAccum.tif' moved to the folder 'GPM_preci